# TASK 2.3: BENCHMARK BASELINE GỐC TRÊN TẬP DEGRADED (MỐC SÀN SỤT GIẢM)
### ĐO ĐẠC MỐC SÀN HIỆU NĂNG FATFORMER TRÊN 6 BIẾN THỂ SUY BIẾN VẬT LÝ & TỔNG HỢP MA TRẬN SỤT GIẢM

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Môi trường**: Google Colab GPU NVIDIA Tesla T4 16GB VRAM  
> **Tài nguyên kế thừa (Rule 4)**:
> - Checkpoint gốc: `/content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth` (Task 1.2)
> - Backbone: `/content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt`
> - Tập kiểm thử suy biến: `/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar` (**19.55 GB** từ Task 2.1)
> - Mốc trần Clean tham chiếu: `/content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv` (**96.45% ACC** từ Task 2.2)
> **Đầu ra nghiệm thu**:
> - Tệp ma trận suy giảm: `/content/drive/MyDrive/Fatformer/log/baseline_degraded_results.csv`
> - Báo cáo chi tiết: `/content/drive/MyDrive/Fatformer/log/baseline_degraded_results.md`
> - Biểu đồ trực quan: `/content/drive/MyDrive/Fatformer/log/baseline_degraded_chart.png`

---

### 1. QUY CHUẨN 6 BIẾN THỂ SUY BIẾN ĐÁNH GIÁ (DEGRADATION SPECIFICATIONS)
1. **`jpeg_q30`**: Nén JPEG sâu $Q=30$ (Artifact khối $8 \times 8$ cường độ cao - Tử huyệt của CLIP)
2. **`jpeg_q50`**: Nén JPEG trung bình $Q=50$ (Chuẩn chia sẻ web/mạng xã hội)
3. **`jpeg_q70`**: Nén JPEG nhẹ $Q=70$
4. **`blur_s1`**: Gaussian Blur bán kính $\sigma = 1.0$
5. **`blur_s2`**: Gaussian Blur bán kính $\sigma = 2.0$
6. **`down_up`**: Downsample $224 \rightarrow 112$ rồi Upsample lại $224$ (Bicubic)

---

### 2. PHƯƠNG PHÁP ĐÁNH GIÁ: FAST-EVAL 6 BIẾN THỂ (PHƯƠNG ÁN 1)
- Lấy mẫu ngẫu nhiên đại diện **500 ảnh/subset** (250 real + 250 fake, cố định `seed=42`).
- Tổng cộng $6 \times 18 \times 500 = 54,000$ ảnh đánh giá.
- Thời gian ước tính trên Colab GPU Tesla T4: **~35 phút**.
- Tự động tính toán mức sụt giảm: $\Delta_{\text{Drop}} = \text{ACC}_{\text{Clean}} - \text{ACC}_{\text{Degraded}}$.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT}. Vui lòng kiểm tra lại cấu trúc Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ mã nguồn Git nhánh hoang
%cd /content
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ branch hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
else:
    print("[*] Đang cập nhật mã nguồn mới nhất từ branch hoang...")
    %cd {REPO_DIR}
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

# Cài đặt các thư viện bổ trợ cần thiết
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn

In [ ]:
# ================================================================================
# BƯỚC 2: KIỂM TRA MÔI TRƯỜNG PHẦN CỨNG GPU T4 & TỆP TRỌNG SỐ BASELINE
# ================================================================================
import torch, shutil

assert torch.cuda.is_available(), "LỖI: Cần kích hoạt GPU trong Runtime -> Change runtime type -> T4 GPU!"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[✓] Thiết bị GPU phát hiện: {gpu_name} ({gpu_mem:.2f} GB VRAM)")

# Đồng bộ checkpoint và ViT-L-14.pt từ Drive vào thư mục repo nếu chưa có
CKPT_DRIVE = f"{DRIVE_ROOT}/pretrained/fatformer_4class_ckpt.pth"
CLIP_DRIVE = f"{DRIVE_ROOT}/pretrained/ViT-L-14.pt"

assert os.path.exists(CKPT_DRIVE), f"LỖI: Không tìm thấy checkpoint tại {CKPT_DRIVE}!"
assert os.path.exists(CLIP_DRIVE), f"LỖI: Không tìm thấy ViT-L-14.pt tại {CLIP_DRIVE}!"

CKPT_LOCAL = "/content/fatformer-xla/fatformer_4class_ckpt.pth"
CLIP_LOCAL = "/content/fatformer-xla/ViT-L-14.pt"

if not os.path.exists(CKPT_LOCAL):
    print(f"[*] Đang sao chép checkpoint sang local SSD...")
    shutil.copy2(CKPT_DRIVE, CKPT_LOCAL)
print(f"[✓] Checkpoint sẵn sàng: {CKPT_LOCAL} ({os.path.getsize(CKPT_LOCAL)/(1024**2):.2f} MB)")

if not os.path.exists(CLIP_LOCAL):
    print(f"[*] Đang sao chép ViT-L-14.pt sang local SSD...")
    shutil.copy2(CLIP_DRIVE, CLIP_LOCAL)
print(f"[✓] Backbone ViT-L-14 sẵn sàng: {CLIP_LOCAL} ({os.path.getsize(CLIP_LOCAL)/(1024**2):.2f} MB)")

# Kiểm tra file Clean baseline tham chiếu từ Task 2.2
CLEAN_CSV_PATH = f"{DRIVE_ROOT}/log/baseline_clean_results.csv"
if os.path.exists(CLEAN_CSV_PATH):
    print(f"[✓] Phát hiện tệp Clean tham chiếu từ Task 2.2: {CLEAN_CSV_PATH}")
else:
    print(f"[!] Không thấy {CLEAN_CSV_PATH} trên Drive, script sẽ dùng mốc tham chiếu mặc định (96.45% Overall).")

In [ ]:
# ================================================================================
# BƯỚC 3: GIẢI NÉN BỘ DỮ LIỆU SUY BIẾN TEST_DEGRADED.TAR SANG SSD NVME MÁY ẢO
# ================================================================================
import os, time, shutil

TAR_PATH = f"{DRIVE_ROOT}/datasets/test_degraded.tar"
LOCAL_EXTRACT_BASE = "/content/dataset_local"
LOCAL_DEGRADED_DIR = f"{LOCAL_EXTRACT_BASE}/test_degraded"

assert os.path.exists(TAR_PATH), f"LỖI: Không tìm thấy tệp {TAR_PATH} trên Drive 5TB!"
tar_size_gb = os.path.getsize(TAR_PATH) / (1024**3)
print(f"[✓] Phát hiện tệp suy biến đóng gói từ Task 2.1: {TAR_PATH} ({tar_size_gb:.2f} GB)")

os.makedirs(LOCAL_EXTRACT_BASE, exist_ok=True)

# Kiểm tra xem đã giải nén chưa (an toàn không giải nén lại nếu đã có)
expected_variants = ['jpeg_q30', 'jpeg_q50', 'jpeg_q70', 'blur_s1', 'blur_s2', 'down_up']
is_extracted = os.path.exists(LOCAL_DEGRADED_DIR) and all(
    os.path.isdir(os.path.join(LOCAL_DEGRADED_DIR, v)) for v in expected_variants
)

if is_extracted:
    print(f"[✓] Thư mục suy biến đã được giải nén sẵn tại: {LOCAL_DEGRADED_DIR}. Bỏ qua bước giải nén!")
else:
    print(f"[*] Đang giải nén {TAR_PATH} sang SSD NVMe máy ảo ({LOCAL_EXTRACT_BASE})...")
    t0 = time.time()
    !tar -xf {TAR_PATH} -C {LOCAL_EXTRACT_BASE}
    elapsed = time.time() - t0
    print(f"[✓] Giải nén hoàn tất sau {elapsed:.2f} giây ({elapsed/60:.2f} phút)!")

# Xác minh cấu trúc 6 biến thể
assert os.path.exists(LOCAL_DEGRADED_DIR), f"LỖI: Không tìm thấy thư mục {LOCAL_DEGRADED_DIR}!"
variants_found = sorted([d for d in os.listdir(LOCAL_DEGRADED_DIR) if os.path.isdir(os.path.join(LOCAL_DEGRADED_DIR, d))])
print(f"\n[✓] Danh sách các biến thể suy biến hiện diện ({len(variants_found)}/6):")
for v in variants_found:
    sub_cnt = len([s for s in os.listdir(os.path.join(LOCAL_DEGRADED_DIR, v)) if os.path.isdir(os.path.join(LOCAL_DEGRADED_DIR, v, s))])
    print(f"  - {v:<12} -> {sub_cnt} subsets chuẩn")

In [ ]:
# ================================================================================
# BƯỚC 4: THỰC THI BENCHMARK BASELINE TRÊN 6 BIẾN THỂ SUY BIẾN (TASK 2.3)
# ================================================================================
# Đánh giá 500 ảnh đại diện / subset (250 real + 250 fake, seed=42)
# Thời gian dự kiến: ~35 phút trên GPU Tesla T4
# Tự động đối chiếu Clean Baseline và lưu kết quả CSV & Markdown lên Drive 5TB

%cd /content/fatformer-xla

OUTPUT_CSV = f"{DRIVE_ROOT}/log/baseline_degraded_results.csv"
OUTPUT_MD = f"{DRIVE_ROOT}/log/baseline_degraded_results.md"
os.makedirs(f"{DRIVE_ROOT}/log", exist_ok=True)

!python tools/eval_degraded.py \
    --checkpoint /content/fatformer-xla/fatformer_4class_ckpt.pth \
    --clip_path /content/fatformer-xla/ViT-L-14.pt \
    --degraded_root /content/dataset_local/test_degraded \
    --variants all \
    --subsets all \
    --samples_per_subset 500 \
    --batch_size 32 \
    --num_workers 4 \
    --clean_csv {CLEAN_CSV_PATH} \
    --output_csv {OUTPUT_CSV} \
    --output_markdown {OUTPUT_MD}

In [ ]:
# ================================================================================
# BƯỚC 5: TẠO BIỂU ĐỒ TRỰC QUAN HÓA MA TRẬN SỤT GIẢM & LƯU LÊN DRIVE 5TB
# ================================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

assert os.path.exists(OUTPUT_CSV), f"LỖI: Chưa có tệp kết quả {OUTPUT_CSV}!"
df = pd.read_csv(OUTPUT_CSV)

# Lấy các dòng tổng kết MEAN_OVERALL của từng biến thể
summary_df = df[df['subset'] == 'MEAN_OVERALL'].copy()
print("[*] Bảng tóm tắt kết quả theo biến thể:")
display(summary_df[['variant', 'variant_desc', 'acc', 'clean_acc', 'delta_drop', 'r_acc', 'f_acc']])

# Thiết lập phong cách đồ họa chuyên nghiệp
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

variants_order = ['Clean', 'jpeg_q70', 'jpeg_q50', 'jpeg_q30', 'blur_s1', 'blur_s2', 'down_up']
clean_acc = float(summary_df['clean_acc'].iloc[0]) if not summary_df.empty else 96.45

chart_data = [{'Variant': 'Clean (Mốc)', 'ACC': clean_acc, 'Type': 'Clean Mốc'}]
for _, row in summary_df.iterrows():
    chart_data.append({
        'Variant': row['variant'],
        'ACC': float(row['acc']),
        'Type': 'Degraded'
    })
chart_df = pd.DataFrame(chart_data)

palette = ['#2ecc71'] + ['#e74c3c' if v == 'jpeg_q30' else '#3498db' for v in chart_df['Variant'][1:]]
ax = sns.barplot(data=chart_df, x='Variant', y='ACC', palette=palette)

plt.title('MỐC SÀN SỤT GIẢM HIỆU NĂNG FATFORMER BASELINE TRÊN 6 BIẾN THỂ SUY BIẾN (TASK 2.3)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Biến Thể Kiểm Thử Vật Lý', fontsize=12, fontweight='bold')
plt.ylabel('Độ Chính Xác ACC (%)', fontsize=12, fontweight='bold')
plt.ylim(0, 105)

# Ghi nhãn giá trị và Delta lên từng cột
for p in ax.patches:
    height = p.get_height()
    delta_text = f"{height - clean_acc:+.1f}%" if height != clean_acc else "Mốc"
    ax.annotate(f"{height:.1f}%\n({delta_text})",
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom',
                fontsize=10, fontweight='bold',
                xytext=(0, 4), textcoords='offset points')

CHART_PATH = f"{DRIVE_ROOT}/log/baseline_degraded_chart.png"
plt.tight_layout()
plt.savefig(CHART_PATH, dpi=300)
plt.show()
print(f"[✓] Đã lưu biểu đồ phân tích trực quan ra: {CHART_PATH}")

In [ ]:
# ================================================================================
# BƯỚC 6: KIỂM ĐỊNH TÍNH TOÀN VẸN CÁC TỆP ĐẦU RA VÀ ĐỒNG BỘ GOOGLE DRIVE
# ================================================================================
import os

print("=" * 80)
print("KIỂM TOÁN TÍNH TOÀN VẸN THÀNH PHẨM NHIỆM VỤ 2.3 (DoD AUDIT GATE)")
print("=" * 80)

files_to_check = [
    (OUTPUT_CSV, "Tệp ma trận dữ liệu định lượng CSV"),
    (OUTPUT_MD, "Báo cáo tổng hợp định dạng Markdown"),
    (f"{DRIVE_ROOT}/log/baseline_degraded_chart.png", "Biểu đồ phân tích sụt giảm PNG")
]

all_passed = True
for fpath, desc in files_to_check:
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        print(f"[✓] ĐẠT: {desc:<40} -> {os.path.basename(fpath)} ({os.path.getsize(fpath)/1024:.1f} KB)")
    else:
        print(f"[!] THIẾU HOẶC RỖNG: {desc:<40} -> {fpath}")
        all_passed = False

print("-" * 80)
if all_passed:
    print("🎉 CHÚC MỪNG: TASK 2.3 ĐÃ HOÀN TẤT XUẤT SẮC 100% TIÊU CHUẨN DoD!")
    print("👉 BƯỚC TIẾP THEO: Điền kết quả vào docs/bao_cao/C/bao_cao_task_2.3_baseline_degraded_benchmark.md")
    print("                 và sẵn sàng cho cuộc họp chốt Milestone 1 (Task 2.5).")
else:
    print("[!] Vui lòng kiểm tra lại bước thực thi đánh giá trước khi nghiệm thu!")
print("=" * 80)

# Đồng bộ bộ nhớ đệm Drive
drive.flush_and_unmount()
print("[✓] Đã đồng bộ an toàn dữ liệu lên Google Drive 5TB!")